# API — Language Judge Run (local / Databricks dual-mode)

First-pass LLM-as-a-Judge evaluation for the API/agent-routing side of Hey George, focused only on **language detection + language compliance**.

This notebook runs the `hg_ds_evals` judge against the API input table written by `experiments/api/notebooks/import_traces_local.ipynb` (`RUN_DATE = hajek_prompt`).

- **YAML config:** [`configs/api_exp_001_language.yaml`](../configs/api_exp_001_language.yaml)
- **System template:** [`configs/system.md.j2`](../configs/system.md.j2)
- **Judge:** `gpt-5-1` via `databricks_async`, `reasoning_effort: medium`
- **Output:** CSV checkpoint under the directory set by `paths.checkpoint_dir` in the YAML (currently `checkpoints/`, relative to the notebook working directory).

## Run modes
- `RUN_ON_DBX = True` — runs on a Databricks cluster, reads the UC table via `spark.read.table(...)`.
- `RUN_ON_DBX = False` — runs on your laptop, reads the UC table via `databricks-sql-connector` against a SQL warehouse, authenticates with the named CLI profile, writes results to the local `checkpoints/` directory.

> **NOTE — why we do not call `run_experiment` directly:** the upstream
> `run_experiment` helper instantiates `PromptBuilder(rubric=rubric)`
> without passing the YAML's `paths.system_template_path`, so the
> default embedded template is used and our `critical_evaluation_rules`,
> `domain_specific_guidance`, and `final_reminders` are silently
> dropped. To keep the custom template in effect we replicate
> `run_experiment`'s flow explicitly below.

Run 
`databricks auth login --host https://adb-3174992876438447.7.azuredatabricks.net` to prevent the auth errors.

## Run mode

In [33]:
RUN_ID:str = "86603de1c02e4474aa136c74915af44e"

In [34]:
import os
# Strip env vars the Databricks VS Code extension injects — its loopback OAuth
# broker URL goes stale on extension/VS Code restarts and hangs the kernel.
# Falling through to the .databrickscfg profile is more robust.
for v in ("DATABRICKS_AUTH_TYPE", "DATABRICKS_METADATA_SERVICE_URL",
          "DATABRICKS_HOST", "DATABRICKS_CLUSTER_ID"):
    os.environ.pop(v, None)
os.environ["DATABRICKS_CONFIG_PROFILE"] = "adb-uat"

In [35]:
# Toggle to run on a Databricks cluster vs. locally from your laptop.
RUN_ON_DBX: bool = False

# Local-only: which Databricks CLI profile to use for auth.
DBX_PROFILE: str = "adb-uat"

DBX_CLUSTER_ID: str | None = "0521-134949-g7yxajzm"
DBX_SQL_WAREHOUSE_ID: str | None = None

# When True, the input dataframe is read from a local CSV instead of
# Unity Catalog. Use this when import_traces_local.ipynb was run with
# WRITE_TO_UC=False, or whenever you'd rather not hit the warehouse.
# The CSV must already contain the columns referenced by the YAML's
# `eval_columns` / `passthrough_columns` (at minimum `user_query` and
# `agent_response` for this experiment).
LOAD_FROM_LOCAL: bool = True

# Path to the local enriched-traces CSV produced by
# experiments/api/notebooks/import_traces_local.ipynb. Ignored when
# LOAD_FROM_LOCAL is False. `~` is expanded.
LOCAL_INPUT_PATH: str = f"~/Developer/input_traces/enriched_traces_{RUN_ID}.csv"

In [36]:
!databricks auth login --profile adb-uat   # OK

]11;?\Profile adb-uat was successfully saved


In [37]:
!databricks auth describe --profile adb-uat   # OK
!databricks current-user me  --profile adb-uat # OK

]11;?\Host: https://adb-7405614616397813.13.azuredatabricks.net
User: sg7cb@s-mxs.net
Authenticated with: databricks-cli
-----
Current configuration:
  ✓ host: https://adb-7405614616397813.13.azuredatabricks.net (from /Users/SG7CB/.databrickscfg config file)
  ✓ account_id: 85cde0e4-68ed-4a99-b884-abaec50d1f90 (from /Users/SG7CB/.databrickscfg config file)
  ✓ workspace_id: 3174992876438447 (from DATABRICKS_WORKSPACE_ID environment variable)
  ✓ profile: adb-uat (from --profile flag)
  ✓ auth_type: databricks-cli (from /Users/SG7CB/.databrickscfg config file)
  ✓ cloud: Azure
  ✓ audience: https://adb-7405614616397813.13.azuredatabricks.net/oidc/v1/token
  ✓ discovery_url: https://adb-7405614616397813.13.azuredatabricks.net/oidc/.well-known/oauth-authorization-server
]11;?\{
  "active":true,
  "displayName":"Cirovic Donev Ita 6028 ED-EXT",
  "emails": [
    {
      "primary":true,
      "type":"work",
      "value":"sg7cb@s-mxs.net"
    }
  ],
  "externalId":"e7491b88-fd90-4b9b-9ce

## Environment & imports

In [38]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [39]:
if RUN_ON_DBX:
    # Cluster-only bootstrap. Skipped locally.
    !pip install -qe /Workspace/Users/sg7cb@s-mxs.net/hey-george-ds/ds_common
    !pip install openai -q
    dbutils.library.restartPython()  # noqa: F821 (provided by DBR)
else:
    print("Local mode — skipping cluster pip installs / restartPython.")
    print("Install once into your venv if missing:")
    print("    uv pip install --system-certs databricks-sql-connector openai")

Local mode — skipping cluster pip installs / restartPython.
Install once into your venv if missing:
    uv pip install --system-certs databricks-sql-connector openai


In [40]:
import os
import sys
from pathlib import Path

# Make the local hg-ds-evals repo importable when running off-cluster.
if not RUN_ON_DBX:
    _repo_root = Path.cwd()
    while _repo_root != _repo_root.parent and not (_repo_root / "hg_ds_evals").is_dir():
        _repo_root = _repo_root.parent
    if (_repo_root / "hg_ds_evals").is_dir() and str(_repo_root) not in sys.path:
        sys.path.insert(0, str(_repo_root))
        print(f'Local repo root added to sys.path: "{_repo_root}"')

    os.environ["DATABRICKS_CONFIG_PROFILE"] = DBX_PROFILE

    from databricks.sdk import WorkspaceClient
    _w = WorkspaceClient()
    print(f"Authenticated as: {_w.current_user.me().user_name}")
    print(f"Workspace host:   {_w.config.host}")

import config_nbs
config_nbs.add_local_paths()

import hg_ds_evals
print(hg_ds_evals.__file__)

import pandas as pd
pd.set_option("display.width", None)

Authenticated as: sg7cb@s-mxs.net
Workspace host:   https://adb-7405614616397813.13.azuredatabricks.net
/Users/SG7CB/Developer/hg-ds-evals/hg_ds_evals/__init__.py


In [41]:
from databricks.sdk import WorkspaceClient
for w in WorkspaceClient().warehouses.list():
    print(w.id, w.state, w.name)

026bb0f57a651258 State.STOPPED Serverless Starter Warehouse
2fb4c78638bff92a State.STOPPED Classic_SQL_warehouse


## UC table reader

Local: `databricks-sql-connector` against a SQL warehouse, authenticating via the same CLI profile.
Cluster: `spark.read.table(...)`.

In [42]:
def _resolve_http_path(workspace_client, *, cluster_id=None, warehouse_id=None) -> tuple[str, str]:
    host = workspace_client.config.host.replace("https://", "").rstrip("/")
    if cluster_id:
        # All-purpose cluster path: /sql/protocolv1/o/<org_id>/<cluster_id>
        org_id = host.split(".")[0].removeprefix("adb-").split(".")[0]
        # Better: read it from the workspace URL: adb-<org_id>.<n>.azuredatabricks.net
        org_id = host.split("adb-")[1].split(".")[0]
        return host, f"/sql/protocolv1/o/{org_id}/{cluster_id}"
    # Warehouse fallback (existing behavior)
    warehouses = list(workspace_client.warehouses.list())
    match = next((w for w in warehouses if w.id == warehouse_id), None) if warehouse_id else None
    if match is None:
        running = [w for w in warehouses if str(w.state) == "RUNNING"]
        match = (running or warehouses)[0]
    return host, f"/sql/1.0/warehouses/{match.id}"


def read_uc_table(table_fqn: str) -> pd.DataFrame:
    """Load the input trace dataframe.

    Three modes, in priority order:
      1. LOAD_FROM_LOCAL=True → read LOCAL_INPUT_PATH (CSV) from disk.
         `table_fqn` is logged but otherwise ignored.
      2. RUN_ON_DBX=True     → Spark `spark.read.table(table_fqn)`.
      3. otherwise           → databricks-sql-connector against a
         cluster / SQL warehouse (existing local-UC behaviour).
    """
    if LOAD_FROM_LOCAL:
        local_path = Path(LOCAL_INPUT_PATH).expanduser()
        assert local_path.exists(), (
            f"LOCAL_INPUT_PATH does not exist: {local_path}. "
            f"Either flip LOAD_FROM_LOCAL=False, or re-run "
            f"import_traces_local.ipynb to produce this CSV."
        )
        print(f"[read_uc_table] LOAD_FROM_LOCAL=True; reading {local_path}")
        print(f"[read_uc_table] (ignoring UC table fqn: {table_fqn})")
        return pd.read_csv(local_path)

    if RUN_ON_DBX:
        return spark.read.table(table_fqn).toPandas()  # noqa: F821 (provided by DBR)

    from databricks import sql as dbx_sql
    server_hostname, http_path = _resolve_http_path(
        _w, cluster_id=DBX_CLUSTER_ID, warehouse_id=DBX_SQL_WAREHOUSE_ID
        )
    # `access_token` callable refreshes OAuth U2M tokens transparently.
    access_token = _w.config.authenticate()["Authorization"].removeprefix("Bearer ")
    with dbx_sql.connect(
        server_hostname=server_hostname,
        http_path=http_path,
        access_token=access_token,
    ) as conn:
        with conn.cursor() as cur:
            cur.execute(f"SELECT * FROM {table_fqn}")
            return cur.fetchall_arrow().to_pandas()

In [43]:
print(RUN_ON_DBX, LOAD_FROM_LOCAL)

False True


## Load the experiment YAML and build the rubric

In [44]:
CHECKPOINT_SUFFIX = "dba_testset100_new"

In [45]:
YAML_FILE_NAME = "api_exp_001_language"
# Custom suffix appended to the checkpoint filename so a new run does not
# overwrite a prior checkpoint. Lands as `..._<reasoning>_<CHECKPOINT_SUFFIX>[_test].csv`.
# Set to "" to keep the old behaviour. Example: "hajek_prompt".


# Databricks CLI profile (~/.databrickscfg) used both for the model-serving
# call and for unattended OAuth token refresh during long runs. Empty string
# defers to the default profile or DATABRICKS_CONFIG_PROFILE env var.
# IMPORTANT: run `databricks auth login --profile <name>` once before the
# run to seed the OAuth refresh token; subsequent token rotation is then
# automatic and does not require browser interaction.
DATABRICKS_PROFILE = "adb-uat"

# When True, drop existing `error=True` rows from the checkpoint file
# before the run starts (with timestamped backup). Lets a re-run cleanly
# retry test cases that failed for transient reasons (e.g. token expiry)
# without accumulating duplicate error rows over retry cycles.
PURGE_ERRORS_BEFORE_RUN = True
config_path = f"../configs/{YAML_FILE_NAME}.yaml"
assert Path(config_path).exists(), f"config file not found: {config_path}"

In [46]:
from hg_ds_evals.rubrics.loader import (
    load_experiment_config,
    build_rubric_from_config,
    get_experiment_name,
)

config = load_experiment_config(config_path)
rubric = build_rubric_from_config(config)

print(f"Experiment       : {config['experiment']['name']}")
print(f"Sample size      : {config['dataset']['test_num_rows']}")
print(f"Rubric name      : {rubric.metadata.name}")
print(f"Rubric persona   : {rubric.metadata.persona}")
print(f"Dimensions       : {rubric.dimension_ids}")
print(f"Input fields     : {rubric.input_field_names}")
print(f"Output fields    : {[f.name for f in rubric.output_schema.fields]}")
print(f"Pass threshold   : {rubric.pass_threshold}")
print(f"Judge model      : {config['model']['model_deployment_name']} "
      f"({config['model']['api_provider']}, reasoning={config['model']['reasoning_effort']})")
print(f"Input table      : {config['dataset']['input_schema']}.{config['dataset']['input_dataset']}")
print(f"test_num_rows    : {config['dataset']['test_num_rows']}")

Experiment       : api_exp_001_language
Sample size      : None
Rubric name      : Custom Rubric
Rubric persona   : You are an expert linguist evaluating the Hey George retail-banking conversational AI. Your job is to (1) identify the language of the user's query, (2) identify the language of the agent's reply, and (3) judge whether the agent answered in the same language the user wrote in.

Dimensions       : ['language_compliance']
Input fields     : ['user_query', 'actual_response']
Output fields    : ['user_query_detected_language', 'actual_response_detected_language', 'language_mismatch_description', 'overall_explanation']
Pass threshold   : 1.4
Judge model      : gpt-5-1 (databricks_async, reasoning=medium)
Input table      : evals_offline_db.dts_eval_ts100_exp_001_8502871b6f55495a8593f3bbeab44fe5
test_num_rows    : None


## Load the input data

In [47]:
input_table = (
    f"{config['dataset']['input_catalog']}."
    f"{config['dataset']['input_schema']}."
    f"{config['dataset']['input_dataset']}"
)
# print(input_table)

df_input = read_uc_table(input_table)
print(f"rows: {len(df_input):,}")
df_input.head()

[read_uc_table] LOAD_FROM_LOCAL=True; reading /Users/SG7CB/Developer/input_traces/enriched_traces_86603de1c02e4474aa136c74915af44e.csv
[read_uc_table] (ignoring UC table fqn: uat_aut_chat00_catalog.evals_offline_db.dts_eval_ts100_exp_001_8502871b6f55495a8593f3bbeab44fe5)
rows: 100


,trace_id,test_case_id,session_id,request_time,execution_duration_ms,state,user_query,eval_domain,eval_persona,expected_agent,...,agent_routing_status,tool_usage_score,tool_usage_rationale,tool_usage_status,tool_usage_correct,tool_usage_incorrect,tool_usage_hallucinated,tool_usage_missing_expected,execution_duration_s,execution_duration_min
0,tr-81e17ed9f1edc664d6e7058bdb0abc55,smoke-100,eval_3871d896d8d94b76b7b3da604fa74958,1781541891870,21535,OK,"Покажи лише кредити, якими я володію.",api,NaN,daily_banking_agent,...,ok,0,Order mismatch: expected ['george-gcg-product_...,ok,[],[],[],[],21.535,0.358917
1,tr-4453b0c13aa73c1be28a64cfc1500d60,smoke-099,eval_7dd1e921bbdf4be18db60a75edcce342,1781541865512,20440,OK,Ktoré účty podporujú prevody medzi mojimi vlas...,api,NaN,daily_banking_agent,...,ok,0,Order mismatch: expected ['george-gcg-product_...,ok,[],[],[],[],20.440,0.340667
2,tr-91f85e810535d7e465f7ec3e8790bf6a,smoke-098,eval_bee0ec8902c84b488b75f3adb633e9a7,1781541837626,22477,OK,Vannak ideiglenes limitek a kártyámon?,api,NaN,daily_banking_agent,...,ok,0,Order mismatch: expected ['george-gcg-product_...,ok,[],[],[],[],22.477,0.374617
3,tr-543e106525426aa64e7dfed1d3d0adfb,smoke-097,eval_ec8955529e174d30adee6472930f173d,1781541800420,31841,OK,Kolik ještě mohu dnes utratit na své kartě?,api,NaN,daily_banking_agent,...,ok,0,Order mismatch: expected ['george-gcg-product_...,ok,[],[],[],[],31.841,0.530683
4,tr-ec94858dbf92d1accb68b3f9c3062ce5,smoke-096,eval_77d54f4d82254dd0bd802abf1b2311f9,1781541774316,20406,OK,"Покажи картки, прив'язані до мого ощадного рах...",api,NaN,daily_banking_agent,...,ok,0,Order mismatch: expected ['george-gcg-product_...,ok,[],[],[],[],20.406,0.340100


## Build the judge's system + user prompts from the **custom** template

We explicitly pass `system_template_path` and `user_template_path` — without this the default embedded template is used and the AGENT CONTEXT / critical_evaluation_rules / final_reminders sections are silently dropped.

In [48]:
from hg_ds_evals.prompts.builder import PromptBuilder
from hg_ds_evals.prompts.common import display_prompt

system_template_path = Path(config["paths"]["system_template_path"])
user_template_path = Path(config["paths"]["user_template_path"])

if not system_template_path.is_absolute():
    system_template_path = (Path.cwd() / system_template_path).resolve()
if not user_template_path.is_absolute():
    user_template_path = (Path.cwd() / user_template_path).resolve()

assert system_template_path.exists(), f"system template missing: {system_template_path}"
assert user_template_path.exists(), f"user template missing: {user_template_path}"
print(f"system template : {system_template_path}")
print(f"user template   : {user_template_path}")

builder = PromptBuilder(
    rubric=rubric,
    system_template_path=system_template_path,
    user_template_path=user_template_path,
)
system_prompt = builder.build_system_prompt()
print(f"system prompt   : {len(system_prompt)} chars")

system template : /Users/SG7CB/Developer/hg-ds-evals/experiments/api/configs/system.md.j2
user template   : /Users/SG7CB/Developer/hg-ds-evals/experiments/api/configs/user.md.j2
system prompt   : 6575 chars


In [49]:
display_prompt(system_prompt, title="System Prompt (API language judge)", font_size=12)

## Preview the user prompt on one real row

In [50]:
sample_row = df_input.head(1).iloc[0].to_dict()

user_prompt = builder.build_user_prompt(sample_row)
display_prompt(user_prompt, title=f"User Prompt — {sample_row.get('test_case_id')}", font_size=12)

## Run the evaluation

The eval results land in the **local** `checkpoints/` directory configured by the YAML (`paths.checkpoint_dir`). On a cluster this is the workspace-mounted notebook dir; locally it's relative to your CWD.

In [52]:
from hg_ds_evals.common.utils import (
    load_checkpoint,
    filter_df_with_checkpoints,
    prepare_eval_sample,
)
from hg_ds_evals.llm.api_client import get_api_client
from hg_ds_evals.evals.evaluator import async_run_evals

experiment_name = get_experiment_name(config_path)
INPUT_CATALOG = config["dataset"]["input_catalog"]
INPUT_SCHEMA = config["dataset"]["input_schema"]
INPUT_TABLE = config["dataset"]["input_dataset"]
ID_COLUMNS = config["dataset"].get("id_columns", [])
NUM_ROWS = config["dataset"]["test_num_rows"]
CHECKPOINT_DIR = config["paths"].get("checkpoint_dir", "checkpoints")

print(f"[1/5] loading input table")
df = read_uc_table(f"{INPUT_CATALOG}.{INPUT_SCHEMA}.{INPUT_TABLE}")
print(f"      rows: {len(df):,}")

print("\n[2/5] preparing eval sample")
df_sample, file_name_eval = prepare_eval_sample(
    df=df,
    evals_name=experiment_name,
    reasoning_effort=config["model"]["reasoning_effort"],
    suffix=CHECKPOINT_SUFFIX or None,
    num_rows=NUM_ROWS,
)
cols = config["dataset"]["eval_columns"]
if ID_COLUMNS:
    cols = list(dict.fromkeys(ID_COLUMNS + cols))
passthrough = config["dataset"].get("passthrough_columns", [])
if passthrough:
    cols = list(dict.fromkeys(cols + passthrough))
# Tolerate columns that are missing locally (e.g. derived columns produced only by Spark notebooks).
missing_cols = [c for c in cols if c not in df_sample.columns]
if missing_cols:
    print(f"      WARN — missing columns dropped: {missing_cols}")
    cols = [c for c in cols if c in df_sample.columns]
df_eval = df_sample[cols].copy()
print(f"      eval df: {len(df_eval)} rows, {len(cols)} cols  checkpoint_file={file_name_eval}")

print("\n[3/5] loading checkpoint")
ckp_df, ckp_path = load_checkpoint(
    checkpoint_file_name=file_name_eval,
    checkpoint_dir=CHECKPOINT_DIR,
    purge_error_rows=PURGE_ERRORS_BEFORE_RUN,
)
df_eval = filter_df_with_checkpoints(df_eval, ckp_df, id_cols=ID_COLUMNS)
print(f"      {len(df_eval)} rows remain after filtering {len(ckp_df)} already-scored rows")
print(f"      checkpoint path: {ckp_path}")

print("\n[4/5] setting up API client")
client = get_api_client(
    model_deployment_name=config["model"]["model_deployment_name"],
    api_provider=config["model"]["api_provider"],
    databricks_endpoint_url=config["model"].get("databricks_endpoint_url"),
    databricks_base_url=config["model"].get("databricks_base_url"),
    databricks_workspace_host=config["model"].get("databricks_workspace_host"),
    databricks_profile=DATABRICKS_PROFILE or None,
)

print("\n[5/5] running evaluations")
results_df, metrics = await async_run_evals(
    df=df_eval,
    system_prompt=system_prompt,
    client=client,
    config=config,
    checkpoint_path=ckp_path,
    user_prompt_builder=builder.build_user_prompt,
)

print("\ndone.")
print(f"checkpoint  : {ckp_path}")
print(f"metrics     : {metrics}")

[1/5] loading input table
[read_uc_table] LOAD_FROM_LOCAL=True; reading /Users/SG7CB/Developer/input_traces/enriched_traces_86603de1c02e4474aa136c74915af44e.csv
[read_uc_table] (ignoring UC table fqn: uat_aut_chat00_catalog.evals_offline_db.dts_eval_ts100_exp_001_8502871b6f55495a8593f3bbeab44fe5)
      rows: 100

[2/5] preparing eval sample
Sample size: 100
File name: evals_api_exp_001_language_medium_dba_testset100_new.csv
      WARN — missing columns dropped: ['user_query_en', 'actual_response_en', 'expected_response_en', 'tool_parameter_score', 'tool_parameter_status']
      eval df: 100 rows, 18 cols  checkpoint_file=evals_api_exp_001_language_medium_dba_testset100_new.csv

[3/5] loading checkpoint
✓ Loading existing checkpoint: checkpoints/evals_api_exp_001_language_medium_dba_testset100_new.csv
ℹ️ Filtered 100 rows from checkpoint
ℹ️ Remaining rows to process: 0
      0 rows remain after filtering 100 already-scored rows
      checkpoint path: checkpoints/evals_api_exp_001_langua

## Check the checkpoint

In [ ]:
df_results = pd.read_csv(ckp_path)
print(f"checkpoint rows: {len(df_results)}")
print()
print("error counts:")
print(df_results.get("error", pd.Series([None]*len(df_results))).isna().value_counts(dropna=False))

dim_score_cols = [c for c in df_results.columns if c.endswith("_score")]
print()
print("dimension score summary:")
print(df_results[dim_score_cols].describe().round(2))

## Next step

The checkpoint CSV contains per-row `language_compliance_score`, the two detected language codes (`user_query_detected_language`, `actual_response_detected_language`), and `language_mismatch_description`. Pivot on those columns to break down agent language drift by user-query language / `eval_domain` / `actual_agent` etc.

When you're happy with the dimension and want to add another judge (e.g. routing-intent alignment, refusal appropriateness, answer alignment vs `expected_response`), extend [`configs/api_exp_001_language.yaml`](../configs/api_exp_001_language.yaml) — or fork it into a new YAML — and re-run this notebook with a different `CHECKPOINT_SUFFIX`.

In [ ]:
df = pd.read_csv(f"checkpoints/evals_{YAML_FILE_NAME}_medium_{CHECKPOINT_SUFFIX}.csv")

In [ ]:
df.language_compliance_score.value_counts(dropna=False)

In [ ]:
df.columns

In [ ]:
df.error.value_counts(dropna=False)

In [ ]:
# Without judge (existing behaviour)
python ../../api_report.py \
  --input ~/Developer/input_traces/enriched_traces_76b28353ceb2474b8aad150d1ce7313f.csv \
  --output reports/api_smoke_report.html

# With judge — adds the Language mismatch card + per-case section
python ../../api_report.py \
  --input ~/Developer/input_traces/enriched_traces_76b28353ceb2474b8aad150d1ce7313f.csv \
  --judge checkpoints/evals_api_exp_001_language_medium_dom_hajek_prompt.csv \
  --output reports/api_smoke_report.html

In [ ]:
!python ../../api_report.py \
  --input    /Users/SG7CB/Developer/input_traces/enriched_traces_61d854d5c01148a883606aed3d1f2c3b.csv \
  --baseline /Users/SG7CB/Developer/input_traces/enriched_traces_d0fafd123f4d47949abcb3d926532085.csv \
  --judge checkpoints/evals_api_exp_001_language_medium_dom_hajek_prompt.csv \
  --output   reports/api_smoke_report_dom_filip_vs_gui_changes.html

## Run the judge on a single row (no checkpoint)

Quick prompt-iteration loop for **one** row: send its system + user prompt straight to the judge and show the parsed scores, the output fields, and the raw JSON — **nothing is written to `checkpoints/`**.

`run_judge_on_row(...)` **reloads the experiment YAML, rebuilds the rubric, and re-reads the `system.md.j2` / `user.md.j2` templates from disk on every call**, so the loop is simply: edit the YAML or a prompt → re-run the cell → read the result.

Select the row any of these ways:
- positional index — `run_judge_on_row(0)` (also accepts a numpy index, e.g. `df_input[df_input.test_case_id == "smoke-004"].index[0]`),
- `test_case_id` string — `run_judge_on_row("smoke-004")`,
- nothing — `run_judge_on_row()` reuses `sample_row` from the preview section.

Pass `show_prompts=True` to also render the exact prompts that were sent. The judge client is cached and only rebuilt when the YAML's `model` section changes.

In [73]:
# ── Single-row judge runner (no checkpoint) ──────────────────────────────────
# Replicates ONE iteration of `async_run_evals` (build user prompt -> single
# async LLM call -> parse_single_row_response) but skips all checkpoint I/O.
# Reloads the YAML + rubric + .j2 templates from disk on every call, so it's a
# fast loop for testing prompt changes. Nothing is written to checkpoints/.
import asyncio

from hg_ds_evals.rubrics.loader import load_experiment_config, build_rubric_from_config
from hg_ds_evals.prompts.builder import PromptBuilder
from hg_ds_evals.llm.api_calls import async_call_llm_for_evaluation
from hg_ds_evals.evals.parsers import parse_single_row_response

# Cache one client across re-runs so we don't re-auth on every iteration; it is
# rebuilt only when the YAML's `model` section changes (or force_new=True).
_SINGLE_ROW_CLIENT: dict = {}


def _single_row_client(model_cfg: dict, force_new: bool = False):
    key = (
        model_cfg["model_deployment_name"],
        model_cfg["api_provider"],
        model_cfg.get("databricks_endpoint_url"),
        model_cfg.get("databricks_base_url"),
        model_cfg.get("databricks_workspace_host"),
    )
    if force_new or _SINGLE_ROW_CLIENT.get("key") != key:
        _SINGLE_ROW_CLIENT["key"] = key
        _SINGLE_ROW_CLIENT["client"] = get_api_client(
            model_deployment_name=model_cfg["model_deployment_name"],
            api_provider=model_cfg["api_provider"],
            databricks_endpoint_url=model_cfg.get("databricks_endpoint_url"),
            databricks_base_url=model_cfg.get("databricks_base_url"),
            databricks_workspace_host=model_cfg.get("databricks_workspace_host"),
            databricks_profile=DATABRICKS_PROFILE or None,
        )
    return _SINGLE_ROW_CLIENT["client"]


def _resolve_row(row) -> dict:
    """Return a plain row dict from None / dict / Series / index / test_case_id.

      * None                      -> ``sample_row`` (the preview row)
      * dict / pandas Series      -> used as-is
      * str                       -> matched against ``df_input.test_case_id``
      * anything int-like         -> positional ``df_input.iloc[int(row)]``
                                     (covers Python int AND numpy.int64, e.g.
                                     ``df_input[mask].index[0]``)
    """
    if row is None:
        return dict(sample_row)
    if isinstance(row, pd.Series):
        return row.to_dict()
    if isinstance(row, dict):
        return row
    if isinstance(row, str):
        match = df_input[df_input["test_case_id"] == row]
        if match.empty:
            raise KeyError(f"No df_input row with test_case_id == {row!r}")
        return match.iloc[0].to_dict()
    return df_input.iloc[int(row)].to_dict()


def _display_single_result(parsed: dict, raw_text, case_id=None) -> None:
    from IPython.display import display, Markdown

    display(Markdown("### Judge result" + (f" — `{case_id}`" if case_id else "")))

    if parsed.get("error"):
        display(Markdown(f"**❌ {parsed.get('error_type')}** — {parsed.get('error_message')}"))
        if raw_text:
            display_prompt(str(raw_text), title="Raw model output", font_size=12)
        return

    # Dimension scores + reasoning (cols ending in _score / _reasoning).
    score_rows = [
        {"dimension": k[:-6], "score": parsed.get(k), "reasoning": parsed.get(f"{k[:-6]}_reasoning")}
        for k in parsed if k.endswith("_score")
    ]
    if score_rows:
        display(Markdown("**Scores**"))
        display(pd.DataFrame(score_rows))

    # Output-schema fields (everything that isn't a score/reasoning/bookkeeping col).
    skip = {"error", "error_type", "error_message", "raw_output_text"}
    field_rows = [
        {"field": k, "value": v}
        for k, v in parsed.items()
        if k not in skip and not k.endswith("_score") and not k.endswith("_reasoning")
    ]
    if field_rows:
        display(Markdown("**Output fields**"))
        display(pd.DataFrame(field_rows))

    display_prompt(str(raw_text), title="Raw model output (JSON)",
                   font_size=12, background_color="#eef7e6")


async def run_judge_on_row(row=None, *, show_prompts: bool = False, verbose: bool = True) -> dict:
    """Run the LLM-as-Judge on a SINGLE row — no checkpoint, results shown inline.

    Reloads the experiment YAML, rebuilds the rubric, and re-reads the
    system.md.j2 / user.md.j2 templates from disk on every call, so edits to the
    YAML or the templates take effect on a plain re-run.

    Args:
        row: a row dict / pandas Series, a positional index into ``df_input``
            (Python int or numpy int), or a ``test_case_id`` string. Defaults to
            ``sample_row`` (the row picked in the preview section).
        show_prompts: also render the exact system + user prompts that were sent.
        verbose: print a one-line note confirming the YAML/templates were reloaded.

    Returns:
        The parsed result dict (output fields + ``*_score`` / ``*_reasoning``),
        as produced by ``parse_single_row_response``.
    """
    row = _resolve_row(row)

    # Always reload the experiment YAML + rubric + templates from disk.
    cfg = load_experiment_config(config_path)
    builder_ = PromptBuilder(
        rubric=build_rubric_from_config(cfg),
        system_template_path=system_template_path,
        user_template_path=user_template_path,
    )
    system_prompt_ = builder_.build_system_prompt()
    if verbose:
        print(f"[reloaded] {config_path}  |  system prompt: {len(system_prompt_):,} chars  |  "
              f"row: {row.get('test_case_id')}  |  model: {cfg['model']['model_deployment_name']} "
              f"(reasoning={cfg['model'].get('reasoning_effort')})")

    if show_prompts:
        display_prompt(system_prompt_, title="System Prompt (sent)", font_size=12)
        display_prompt(builder_.build_user_prompt(row),
                       title=f"User Prompt (sent) — {row.get('test_case_id')}", font_size=12)

    # Single async call — semaphore of 1, no batching, no checkpoint.
    result = await async_call_llm_for_evaluation(
        row, _single_row_client(cfg["model"]), asyncio.Semaphore(1), system_prompt_, cfg,
        user_prompt_builder=builder_.build_user_prompt,
    )

    # Normalise into the shape parse_single_row_response expects (mirrors evaluator).
    result_dict: dict = {}
    usage = None
    if isinstance(result, dict) and "error" in result:
        result_dict.update(output_text=None, error=True,
                           error_type=result.get("error_type", "EvaluationError"),
                           error_message=result.get("error", ""))
    else:
        result_dict.update(output_text=result.output_text, error=False)
        usage = getattr(result, "usage", None)

    parsed = parse_single_row_response(result_dict, id_columns=[], passthrough_columns=[])

    _display_single_result(parsed, result_dict.get("output_text"), case_id=row.get("test_case_id"))
    if usage is not None:
        print(f"tokens — input: {usage.input_tokens:,}  "
              f"output: {usage.output_tokens:,}  total: {usage.total_tokens:,}")

    return parsed

In [96]:
test_case_name = "smoke-088"
cond1 = df_input.test_case_id == test_case_name
ids = df_input[cond1].index[0]
print(ids)

12


In [97]:
# Pick which df_input row to judge (the preview section above used row 0).
# Iterate on prompts: edit configs/system.md.j2, configs/user.md.j2, or the
# rubric in the YAML, then just re-run this cell — nothing is checkpointed.
SINGLE_ROW_INDEX = ids

result_single = await run_judge_on_row(SINGLE_ROW_INDEX, show_prompts=False)

[reloaded] ../configs/api_exp_001_language.yaml  |  system prompt: 6,625 chars  |  row: smoke-088  |  model: gpt-5-1 (reasoning=medium)


### Judge result — `smoke-088`

**Scores**

,dimension,score,reasoning
0,language_compliance,1,The user wrote in Ukrainian and the agent resp...


**Output fields**

,field,value
0,user_query_detected_language,uk
1,actual_response_detected_language,uk
2,language_mismatch_description,Response is mainly Ukrainian but contains seve...
3,overall_explanation,"The user query is in Ukrainian, and the agent’..."


tokens — input: 1,898  output: 409  total: 2,307


In [100]:
from decimal import Decimal
Decimal("2.5") == 2.5

True